# M1: Explore the corpus

Re-measures every fact in `PLAN.md` section 2 with code, then gathers the numbers behind the
ingest rules (dates, duplicates, quoted text, threads, months observed).

* Code only, no model calls. Standard library only.
* Email text is untrusted data: this notebook prints counts and, at most, short snippets.
* `corpus/` is read, never written.

Each fact is recorded with `check(...)`, and the last cell prints a table of
PLAN value vs measured value.

In [1]:
import csv, hashlib, re, statistics, collections, datetime as dt, math
from pathlib import Path
from email.parser import HeaderParser
from email.utils import parsedate_tz, mktime_tz, parsedate_to_datetime, getaddresses

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "corpus" / "MANIFEST.csv").exists())
CORPUS = ROOT / "corpus"
FACTS = []

def check(fact, plan, measured, ok=None):
    if ok is None:
        ok = plan == measured
    FACTS.append((fact, plan, measured, "confirmed" if ok else "CORRECTED"))
    print(f"[{'ok' if ok else 'DIFF'}] {fact}: plan={plan!r} measured={measured!r}")

def pct(a, b):
    return round(100 * a / b, 1) if b else 0.0

def month_index(d):
    return d.year * 12 + d.month - 1

print(ROOT)

C:\Users\Xigua\GitHub\observation_agent


## 1. Files, size, tokens, manifest

In [2]:
manifest = list(csv.DictReader(open(CORPUS / "MANIFEST.csv", newline="", encoding="utf-8")))
files = sorted(p for p in CORPUS.rglob("*") if p.is_file() and p.suffix == ".txt")
non_email = sorted(p.name for p in CORPUS.rglob("*") if p.is_file() and p.suffix != ".txt")
print("non-email files in corpus/:", non_email)

def key_of(path):
    rel = path.relative_to(CORPUS).with_suffix("")
    return "/".join(rel.parts)

def sort_key(k):
    a, b, n = k.split("/")
    return (a, b, int(n))

man_by_new = {r["new_path"]: r for r in manifest}
local_paths = {p.relative_to(ROOT).as_posix() for p in files}
print("manifest rows:", len(manifest), "| files on disk:", len(files))
print("manifest == disk:", set(man_by_new) == local_paths)
print("all original names end in '.':", all(r["original_path"].endswith(".") and not r["original_path"].endswith("..") for r in manifest))
print("original == new minus 'txt':", all(r["new_path"] == r["original_path"] + "txt" for r in manifest))

raw = {}
bad_hash, bad_size = [], []
for p in files:
    b = p.read_bytes()
    k = key_of(p)
    raw[k] = b
    m = man_by_new[p.relative_to(ROOT).as_posix()]
    if hashlib.sha256(b).hexdigest() != m["sha256"]: bad_hash.append(k)
    if len(b) != int(m["bytes"]): bad_size.append(k)
print("SHA-256 mismatches:", len(bad_hash), "| size mismatches:", len(bad_size))

total = sum(len(b) for b in raw.values())
sizes = sorted(len(b) for b in raw.values())
check("file count", 3240, len(raw))
check("total size (MB, 10^6 bytes)", 6.85, round(total / 1e6, 2))
print("total size MiB:", round(total / 2**20, 2))
check("token estimate (chars/4, millions)", 1.7, round(total / 4 / 1e6, 2), ok=abs(total / 4 / 1e6 - 1.7) < 0.05)
check("median email size (KB, 10^3)", 1.4, round(statistics.median(sizes) / 1e3, 1), ok=abs(statistics.median(sizes) / 1e3 - 1.4) < 0.1)
print("size percentiles (bytes): p10 %d p50 %d p90 %d p99 %d max %d" % (tuple(sizes[int(len(sizes) * q)] for q in (0.1, 0.5, 0.9, 0.99)) + (sizes[-1],)))

non-email files in corpus/: ['.gitattributes', 'ABOUT.md', 'MANIFEST.csv']


manifest rows: 3240 | files on disk: 3240
manifest == disk: True
all original names end in '.': True
original == new minus 'txt': True


SHA-256 mismatches: 0 | size mismatches: 0
[ok] file count: plan=3240 measured=3240
[ok] total size (MB, 10^6 bytes): plan=6.85 measured=6.85
total size MiB: 6.53
[ok] token estimate (chars/4, millions): plan=1.7 measured=1.71
[ok] median email size (KB, 10^3): plan=1.4 measured=1.4
size percentiles (bytes): p10 692 p50 1423 p90 3562 p99 10588 max 350222


In [3]:
folders = collections.Counter("/".join(k.split("/")[:2]) for k in raw)
for f, n in sorted(folders.items()):
    print(f"{f:28s} {n:5d}")
plan_folders = {"farmer-d/logistics": 1194, "giron-d/inbox": 380, "giron-d/sent": 789, "lokay-m/inbox": 57,
                "lokay-m/sent_items": 161, "williams-w3/inbox": 139, "williams-w3/sent_items": 520}
check("folder counts", plan_folders, dict(sorted(folders.items())))

farmer-d/logistics            1194
giron-d/inbox                  380
giron-d/sent                   789
lokay-m/inbox                   57
lokay-m/sent_items             161
williams-w3/inbox              139
williams-w3/sent_items         520
[ok] folder counts: plan={'farmer-d/logistics': 1194, 'giron-d/inbox': 380, 'giron-d/sent': 789, 'lokay-m/inbox': 57, 'lokay-m/sent_items': 161, 'williams-w3/inbox': 139, 'williams-w3/sent_items': 520} measured={'farmer-d/logistics': 1194, 'giron-d/inbox': 380, 'giron-d/sent': 789, 'lokay-m/inbox': 57, 'lokay-m/sent_items': 161, 'williams-w3/inbox': 139, 'williams-w3/sent_items': 520}


## 2. Line endings and encoding

In [4]:
crlf_only = lf_only = mixed = cr_bare = 0
for b in raw.values():
    n_crlf = b.count(b"\r\n")
    n_lf_bare = b.count(b"\n") - n_crlf
    if b.replace(b"\r\n", b"").count(b"\r"): cr_bare += 1
    if n_crlf and n_lf_bare: mixed += 1
    elif n_crlf: crlf_only += 1
    else: lf_only += 1
print(f"CRLF only {crlf_only} | LF only {lf_only} | mixed {mixed} | files with bare CR {cr_bare}")
check("files with mixed line endings", 3135, mixed)
print(".gitattributes:", (CORPUS / ".gitattributes").read_text().strip() if (CORPUS / ".gitattributes").exists() else "MISSING")

non_ascii = [k for k, b in raw.items() if any(c > 127 for c in b)]
bad_utf8 = []
for k, b in raw.items():
    try: b.decode("utf-8")
    except UnicodeDecodeError: bad_utf8.append(k)
print("files with non-ASCII bytes:", len(non_ascii), "| not valid UTF-8:", len(bad_utf8), bad_utf8[:5])

CRLF only 96 | LF only 9 | mixed 3135 | files with bare CR 0
[ok] files with mixed line endings: plan=3135 measured=3135
.gitattributes: # Keep the email files byte-for-byte identical to the originals (no line-ending conversion).
* -text
files with non-ASCII bytes: 0 | not valid UTF-8: 0 []


## 3. Parse headers and bodies

In [5]:
SPLIT = re.compile(r"\r?\n\r?\n")
msgs = {}
for k in sorted(raw, key=sort_key):
    text = raw[k].decode("utf-8", errors="replace")
    m = SPLIT.search(text)
    head, body = (text[:m.start()], text[m.end():]) if m else (text, "")
    h = HeaderParser().parsestr(head)
    msgs[k] = dict(
        key=k, mailbox=k.split("/")[0], folder="/".join(k.split("/")[:2]), size=len(raw[k]),
        message_id=h.get("Message-ID"), date_raw=h.get("Date"),
        frm=(h.get("From") or "").strip().lower(),
        to=[a.lower() for _, a in getaddresses(h.get_all("To") or []) if a],
        cc=[a.lower() for _, a in getaddresses(h.get_all("Cc") or []) if a],
        subject=re.sub(r"\s+", " ", h.get("Subject") or "").strip(),
        headers=set(h.keys()), body=body,
    )
print("parsed:", len(msgs))
print("messages with no blank line after headers:", sum(1 for k in raw if not SPLIT.search(raw[k].decode('utf-8', 'replace'))))
print("threading headers present (In-Reply-To / References):",
      sum(1 for m in msgs.values() if {"In-Reply-To", "References"} & m["headers"]))
print("empty bodies (after whitespace strip):", sum(1 for m in msgs.values() if not m["body"].strip()))

hdr_bytes = sum(m["size"] - len(m["body"]) for m in msgs.values())
print(f"header bytes: {hdr_bytes:,} = {pct(hdr_bytes, total)}% of the corpus")
n_rcpt = sorted(len(m["to"]) + len(m["cc"]) for m in msgs.values())
print("recipients (To+Cc) per message: p50", n_rcpt[len(n_rcpt) // 2], "p90", n_rcpt[int(.9 * len(n_rcpt))], "p99", n_rcpt[int(.99 * len(n_rcpt))], "max", n_rcpt[-1])
print("messages with > 25 recipients:", sum(1 for n in n_rcpt if n > 25))

parsed: 3240
messages with no blank line after headers: 0
threading headers present (In-Reply-To / References): 0
empty bodies (after whitespace strip): 0
header bytes: 2,663,840 = 38.9% of the corpus
recipients (To+Cc) per message: p50 1 p90 12 p99 67 max 595
messages with > 25 recipients: 89


## 4. Dates

In [6]:
# The stdlib trap: mktime_tz/timegm silently roll impossible fields over (Feb 32 -> Mar 4),
# while building a datetime from the fields raises. The rule must validate the fields.
t = parsedate_tz("Thu, 32 Feb 2001 25:61:00 -0800")
print("parsedate_tz('32 Feb 2001 25:61'):", t)
if t: print("  mktime_tz would give:", dt.datetime.fromtimestamp(mktime_tz(t), dt.timezone.utc))
try: parsedate_to_datetime("Thu, 32 Feb 2001 25:61:00 -0800")
except Exception as e: print("  datetime(...) raises:", type(e).__name__, e)

# Second trap (found 21 Nov while reviewing outside test code): parsedate_tz treats any year below 100
# as a two-digit year, so the literal year 0001 silently becomes 2001.
print("parsedate_tz('Sat, 26 Nov 0001 ...') year:", parsedate_tz("Sat, 26 Nov 0001 07:35:14 -0800")[0])

WINDOW = (1999, 2002)
DATE_HEAD = re.compile(r"^\s*(?:([A-Za-z]{3,}),\s*)?(\d{1,2})\s+[A-Za-z]+\s+(\d+)\s")
DAY_NAMES = ["mon", "tue", "wed", "thu", "fri", "sat", "sun"]
# Third trap (found in M2): parsedate_tz reports a missing or unknown zone as offset 0, same as UTC.
ZONE = re.compile(r"\d{1,2}:\d{2}(?::\d{2})?\s+([+-]\d{4}|UT|UTC|GMT|Z|[ECMP][SD]T)\b", re.I)
print("parsedate_tz('Mon, 4 Feb 2002 13:47:02') offset:", parsedate_tz("Mon, 4 Feb 2002 13:47:02")[9], "(no zone written)")

def parse_date(s):
    if s is None or not s.strip():
        return None, "missing"
    t = parsedate_tz(s)
    if t is None:
        return None, "invalid"
    h = DATE_HEAD.match(s)
    year = int(h.group(3)) if h else t[0]          # the year exactly as written, never 'repaired'
    if not WINDOW[0] <= year <= WINDOW[1]:
        return None, "out_of_range"
    try:
        local = dt.datetime(year, *t[1:6])         # raises on day 32, hour 25, minute 61
    except ValueError:
        return None, "invalid"
    if h and h.group(1) and h.group(1)[:3].lower() != DAY_NAMES[local.weekday()]:
        return None, "invalid"                     # the weekday name contradicts the date
    if t[9] is None or not ZONE.search(s):
        return None, "no_timezone"                 # never assume UTC (parsedate_tz reports a missing zone as 0)
    return local.replace(tzinfo=dt.timezone(dt.timedelta(seconds=t[9]))).astimezone(dt.timezone.utc), "ok"

for m in msgs.values():
    m["date_utc"], m["date_status"] = parse_date(m["date_raw"])
status = collections.Counter(m["date_status"] for m in msgs.values())
print(status)
for m in msgs.values():
    if m["date_status"] != "ok":
        print(f"  {m['key']:28s} {m['date_status']:13s} {m['date_raw']!r}")
years = collections.Counter(m["date_utc"].year for m in msgs.values() if m["date_status"] == "ok")
print("good dates by year:", dict(sorted(years.items())))
check("missing Date header", ["giron-d/sent/795"], [m["key"] for m in msgs.values() if m["date_status"] == "missing"])
check("invalid date", ["williams-w3/inbox/142"], [m["key"] for m in msgs.values() if m["date_status"] == "invalid"])
check("out-of-range date", ["lokay-m/inbox/74"], [m["key"] for m in msgs.values() if m["date_status"] == "out_of_range"])
check("good dates by year", {1999: 54, 2000: 1219, 2001: 1497, 2002: 467}, dict(sorted(years.items())))

parsedate_tz('32 Feb 2001 25:61'): (2001, 2, 32, 25, 61, 0, 0, 1, -1, -28800)
  mktime_tz would give: 2001-03-05 10:01:00+00:00
  datetime(...) raises: ValueError day is out of range for month
parsedate_tz('Sat, 26 Nov 0001 ...') year: 2001
parsedate_tz('Mon, 4 Feb 2002 13:47:02') offset: 0 (no zone written)
Counter({'ok': 3236, 'out_of_range': 2, 'missing': 1, 'invalid': 1})
  giron-d/sent/795             missing       None
  lokay-m/inbox/50             out_of_range  'Sat, 26 Nov 0001 07:35:14 -0800 (PST)'
  lokay-m/inbox/74             out_of_range  'Wed, 12 Mar 2031 08:14:00 -0600 (CST)'
  williams-w3/inbox/142        invalid       'Tues, 32 Febuary 2001 25:61:00'
good dates by year: {1999: 54, 2000: 1219, 2001: 1496, 2002: 467}
[ok] missing Date header: plan=['giron-d/sent/795'] measured=['giron-d/sent/795']
[ok] invalid date: plan=['williams-w3/inbox/142'] measured=['williams-w3/inbox/142']
[DIFF] out-of-range date: plan=['lokay-m/inbox/74'] measured=['lokay-m/inbox/50', 'lokay-m

In [7]:
# Year boundaries in local time vs UTC can move a message across a year line; check how many.
local_years = collections.Counter()
for m in msgs.values():
    if m["date_status"] == "ok":
        t = parsedate_tz(m["date_raw"]); local_years[t[0]] += 1
print("by year in the header's own time zone:", dict(sorted(local_years.items())))
tz = collections.Counter(re.findall(r"[+-]\d{4}", m["date_raw"] or "")[0] if re.findall(r"[+-]\d{4}", m["date_raw"] or "") else None for m in msgs.values())
print("time zone offsets:", tz.most_common())
ok_dates = sorted(m["date_utc"] for m in msgs.values() if m["date_status"] == "ok")
print("earliest good:", ok_dates[0], "| latest good:", ok_dates[-1])
print("earliest 3:", [d.date().isoformat() for d in ok_dates[:3]], "| latest 3:", [d.date().isoformat() for d in ok_dates[-3:]])

by year in the header's own time zone: {1999: 54, 2000: 1219, 2001: 1496, 2002: 467}
time zone offsets: [('-0800', 1882), ('-0700', 1355), (None, 2), ('-0600', 1)]
earliest good: 1999-12-13 10:19:00+00:00 | latest good: 2002-03-25 23:13:29+00:00
earliest 3: ['1999-12-13', '1999-12-14', '1999-12-14'] | latest 3: ['2002-03-25', '2002-03-25', '2002-03-25']


## 5. Duplicates

In [8]:
def norm_ws(s):
    return re.sub(r"\s+", " ", s).strip()

for m in msgs.values():
    m["norm_body"] = norm_ws(m["body"])
    m["body_hash"] = hashlib.sha256(m["norm_body"].encode("utf-8")).hexdigest()

by_mid = collections.defaultdict(list)
for m in msgs.values(): by_mid[m["message_id"]].append(m["key"])
mid_groups = [v for v in by_mid.values() if len(v) > 1]
check("extra copies by Message-ID", 1, sum(len(v) - 1 for v in mid_groups))
print("Message-ID groups:", mid_groups)

by_body = collections.defaultdict(list)
for m in msgs.values(): by_body[m["body_hash"]].append(m["key"])
body_groups = [sorted(v, key=sort_key) for v in by_body.values() if len(v) > 1]
extra = sum(len(v) - 1 for v in body_groups)
check("extra copies by normalized body", 49, extra)
print("groups:", len(body_groups), "| group sizes:", collections.Counter(len(v) for v in body_groups))
print("shortest duplicated body (chars):", min(len(msgs[v[0]]["norm_body"]) for v in body_groups))

[ok] extra copies by Message-ID: plan=1 measured=1
Message-ID groups: [['farmer-d/logistics/880', 'williams-w3/inbox/143']]
[ok] extra copies by normalized body: plan=49 measured=49
groups: 42 | group sizes: Counter({2: 41, 9: 1})
shortest duplicated body (chars): 41


In [9]:
# Are body twins really the same message? Compare the other header fields inside each group.
def same(group, field):
    return len({str(msgs[k][field]) for k in group}) == 1

kinds = collections.Counter()
for g in body_groups:
    sig = []
    sig.append("same-from" if same(g, "frm") else "diff-from")
    sig.append("same-subject" if same(g, "subject") else "diff-subject")
    sig.append("same-date" if same(g, "date_raw") else "diff-date")
    sig.append("same-mid" if same(g, "message_id") else "diff-mid")
    sig.append("same-folder" if same(g, "folder") else "cross-folder")
    kinds[" / ".join(sig)] += 1
for s, n in kinds.most_common():
    print(f"{n:3d}  {s}")

# Show headers (no body text) of a few groups, especially any that differ in date or subject.
shown = 0
for g in body_groups:
    if not (same(g, "date_raw") and same(g, "subject")) and shown < 4:
        shown += 1
        print("--- body chars:", len(msgs[g[0]]["norm_body"]))
        for k in g:
            print(f"   {k:28s} {msgs[k]['date_raw']!s:38.38s} {msgs[k]['frm']:28.28s} {msgs[k]['subject'][:40]!r}")

 29  same-from / same-subject / diff-date / diff-mid / same-folder
  5  same-from / same-subject / diff-date / diff-mid / cross-folder
  3  same-from / diff-subject / diff-date / diff-mid / same-folder
  2  diff-from / same-subject / diff-date / diff-mid / cross-folder
  2  same-from / same-subject / same-date / diff-mid / cross-folder
  1  same-from / same-subject / same-date / same-mid / cross-folder
--- body chars: 187
   farmer-d/logistics/184       Wed, 11 Apr 2001 07:21:33 -0700 (PDT)  rickya@calpine.com           'Calpine Daily Gas Nomination'
   farmer-d/logistics/1310      Thu, 27 Jul 2000 00:26:00 -0700 (PDT)  rickya@calpine.com           'Calpine Daily Gas Nomination'
   farmer-d/logistics/1312      Mon, 31 Jul 2000 00:20:00 -0700 (PDT)  rickya@calpine.com           'Calpine Daily Gas Nomination'
   farmer-d/logistics/1316      Wed, 2 Aug 2000 00:19:00 -0700 (PDT)   rickya@calpine.com           'Calpine Daily Gas Nomination'
   farmer-d/logistics/1318      Thu, 3 Aug 2000 00

In [10]:
# Date spread inside each body group: how far apart are the copies?
spreads = []
for g in body_groups:
    ds = [msgs[k]["date_utc"] for k in g if msgs[k]["date_status"] == "ok"]
    if len(ds) > 1:
        spreads.append((max(ds) - min(ds)).total_seconds() / 3600)
spreads.sort()
print("groups with 2+ dated copies:", len(spreads))
print("hours between earliest and latest copy: 0h exactly:", sum(1 for s in spreads if s == 0),
      "| <=24h:", sum(1 for s in spreads if s <= 24), "| >24h:", sum(1 for s in spreads if s > 24))
print("largest spreads (hours):", [round(s, 1) for s in spreads[-5:]])

groups with 2+ dated copies: 42
hours between earliest and latest copy: 0h exactly: 3 | <=24h: 41 | >24h: 1
largest spreads (hours): [11.0, 11.0, 11.0, 18.2, 6198.9]


### 5b. Which body twins are real copies?

The 9-copy group above is a daily notice (`Calpine Daily Gas Nomination`) whose body never changes,
spread over 8 months. Those are 9 separate events of a recurring process, not copies. Look at the
time offset between twins: a copy made by the mail export should differ by a whole number of hours
(time-zone mangling) and otherwise match.

In [11]:
pair_rows = []
for g in body_groups:
    first = msgs[g[0]]
    for k in g[1:]:
        m = msgs[k]
        dh = None
        if first["date_status"] == "ok" and m["date_status"] == "ok":
            dh = abs((m["date_utc"] - first["date_utc"]).total_seconds()) / 3600
        pair_rows.append(dict(a=g[0], b=k, dh=dh, whole=dh is not None and dh == int(dh),
                              same_from=first["frm"] == m["frm"], same_subj=first["subject"] == m["subject"],
                              same_mid=first["message_id"] == m["message_id"],
                              month_cross=dh is not None and month_index(first["date_utc"]) != month_index(m["date_utc"])))
print("pairs (each later copy vs the group's first):", len(pair_rows))
print("offset in hours:", collections.Counter(round(r["dh"], 2) for r in pair_rows if r["dh"] is not None).most_common())
print("whole-hour offsets:", sum(r["whole"] for r in pair_rows), "| not whole:", sum(1 for r in pair_rows if not r["whole"]))
print("\nsubject differences:")
for r in pair_rows:
    if not r["same_subj"]:
        print(f"   {r['a']:26s} {msgs[r['a']]['subject'][:60]!r}\n   {r['b']:26s} {msgs[r['b']]['subject'][:60]!r}")
print("\nsender differences:")
for r in pair_rows:
    if not r["same_from"]:
        print(f"   {r['a']:26s} {msgs[r['a']]['frm']}\n   {r['b']:26s} {msgs[r['b']]['frm']}")
print("\nnon-whole offsets:")
for r in pair_rows:
    if not r["whole"]:
        print(f"   {r['a']:26s} {msgs[r['a']]['date_raw']}\n   {r['b']:26s} {msgs[r['b']]['date_raw']}  ({round(r['dh'], 2)} h)  subj {msgs[r['b']]['subject'][:40]!r}")

pairs (each later copy vs the group's first): 49
offset in hours: [(11.0, 22), (10.0, 6), (0.0, 4), (4.01, 2), (4.0, 2), (6198.93, 1), (6103.03, 1), (6055.04, 1), (6030.96, 1), (5860.06, 1), (5839.08, 1), (5743.13, 1), (5599.06, 1), (18.2, 1), (3.8, 1), (0.35, 1), (0.04, 1), (0.06, 1)]
whole-hour offsets: 33 | not whole: 16

subject differences:
   farmer-d/logistics/184     'Calpine Daily Gas Nomination'
   farmer-d/logistics/1330    'Calpine Daily Gas Nomination (weekend)'
   lokay-m/sent_items/7       'Enron Benefit Phone Numbers'
   lokay-m/sent_items/8       'FW: Enron Benefit Phone Numbers'
   williams-w3/sent_items/497 'FW: Decremental Bidding'
   williams-w3/sent_items/498 'Decremental Bidding'

sender differences:
   giron-d/inbox/18           announcements.enron@enron.com
   williams-w3/inbox/37       no.address@enron.com
   giron-d/inbox/54           coo.jeff@enron.com
   williams-w3/inbox/87       no.address@enron.com

non-whole offsets:
   farmer-d/logistics/184     Wed, 1

In [12]:
# Candidate duplicate rules, scored by how many extra copies they mark.
def mark_dups(is_dup):
    extra = 0
    for g in body_groups:
        kept = [g[0]]
        for k in g[1:]:
            if any(is_dup(msgs[c], msgs[k]) for c in kept): extra += 1
            else: kept.append(k)
    return extra

def dh(a, b):
    if a["date_status"] != "ok" or b["date_status"] != "ok": return None
    return abs((a["date_utc"] - b["date_utc"]).total_seconds()) / 3600

rules = {
    "A body only": lambda a, b: True,
    "B body + same sender": lambda a, b: a["frm"] == b["frm"],
    "C body + sender + within 24 h": lambda a, b: a["frm"] == b["frm"] and dh(a, b) is not None and dh(a, b) <= 24,
    "D body + (same Message-ID or within 24 h)": lambda a, b: a["message_id"] == b["message_id"] or (dh(a, b) is not None and dh(a, b) <= 24),
    "E body + norm. subject + (same Message-ID or within 12 h)": lambda a, b: norm_subj0(a["subject"]) == norm_subj0(b["subject"]) and (a["message_id"] == b["message_id"] or (dh(a, b) is not None and dh(a, b) <= 12)),
}
def norm_subj0(s):
    return norm_ws(re.sub(r"^\s*((re|fw|fwd)\s*(\[\d+\])?\s*:\s*)+", "", s or "", flags=re.I)).lower()
for name, f in rules.items():
    print(f"{name:58s} extra copies = {mark_dups(f)}")
cal = sorted((msgs[k] for k in next(g for g in body_groups if len(g) == 9)), key=lambda m: m["date_utc"])
gaps_cal = sorted(dh(a, b) for a, b in zip(cal, cal[1:]))
same_subj_gaps = sorted(dh(a, b) for a, b in zip(cal, cal[1:]) if a["subject"] == b["subject"])
print("closest two Calpine notices (hours):", round(gaps_cal[0], 2), "| closest with the same subject:", round(same_subj_gaps[0], 2))

# Proposed rule E: build the duplicate map used by the rest of the notebook.
dup_of = {}
for g in body_groups:
    kept = [g[0]]
    for k in g[1:]:
        hit = next((c for c in kept if rules["E body + norm. subject + (same Message-ID or within 12 h)"](msgs[c], msgs[k])), None)
        if hit: dup_of[k] = hit
        else: kept.append(k)
print("rule E duplicates:", len(dup_of), "| offsets (h):", sorted(collections.Counter(round(dh(msgs[k], msgs[v]), 1) for k, v in dup_of.items()).items()))
print("rule E kept separate (same body, different event):", [k for g in body_groups for k in g[1:] if k not in dup_of])
print("undated messages inside any body group:", [k for g in body_groups for k in g if msgs[k]["date_status"] != "ok"])
print("rule E pairs whose two dates fall in different calendar months:",
      sum(1 for k, v in dup_of.items() if month_index(msgs[k]["date_utc"]) != month_index(msgs[v]["date_utc"])))
print("rule E pairs in different mailboxes:", sum(1 for k, v in dup_of.items() if msgs[k]["mailbox"] != msgs[v]["mailbox"]))

A body only                                                extra copies = 49
B body + same sender                                       extra copies = 47
C body + sender + within 24 h                              extra copies = 40
D body + (same Message-ID or within 24 h)                  extra copies = 42
E body + norm. subject + (same Message-ID or within 12 h)  extra copies = 40
closest two Calpine notices (hours): 20.98 | closest with the same subject: 24.08
rule E duplicates: 40 | offsets (h): [(0.0, 5), (0.1, 1), (0.3, 1), (3.8, 1), (4.0, 4), (10.0, 6), (11.0, 22)]
rule E kept separate (same body, different event): ['farmer-d/logistics/1310', 'farmer-d/logistics/1312', 'farmer-d/logistics/1316', 'farmer-d/logistics/1318', 'farmer-d/logistics/1329', 'farmer-d/logistics/1330', 'farmer-d/logistics/1336', 'farmer-d/logistics/1348', 'williams-w3/inbox/37']
undated messages inside any body group: []
rule E pairs whose two dates fall in different calendar months: 0
rule E pairs in diffe

## 6. The giant message

In [13]:
G = "farmer-d/logistics/3221"
g = msgs[G]
print("bytes:", g["size"], "| token estimate:", g["size"] // 4, "| share of corpus bytes:", pct(g["size"], total), "%")
print("next largest:", [(k, msgs[k]["size"]) for k in sorted(msgs, key=lambda k: -msgs[k]["size"])[1:4]])
ROW = re.compile(r"^\s*(\d{7})\s+(\S+)\s+(\S+)\s+(\d{1,2}/\s?\d{1,2}/\d{4})\s+([\d,]+)\s+([\d,]+)\s+([+-][\d,]+)\s+(\S+)\s+(\S+)\s*$")
lines = g["body"].splitlines()
rows = [ROW.match(l) for l in lines]
rows = [r for r in rows if r]
other = [l for l in lines if l.strip() and not ROW.match(l)]
check("giant message table rows", 3600, len(rows))
print("non-row, non-blank lines:", len(other))
for l in other: print("   ", l[:110])   # the prose + table header (short)

fd = [tuple(int(x) for x in r.group(4).replace(" ", "").split("/")) for r in rows]
def valid(y, mo, d):
    try: dt.date(y, mo, d); return True
    except ValueError: return False
md_invalid = sum(1 for a, b, y in fd if not valid(y, a, b))   # read as month/day
dm_invalid = sum(1 for a, b, y in fd if not valid(y, b, a))   # read as day/month
ambiguous = sum(1 for a, b, y in fd if valid(y, a, b) and valid(y, b, a) and a != b)
print(f"FLOW_DATE as month/day: {md_invalid} invalid of {len(fd)} | as day/month: {dm_invalid} invalid | ambiguous (both valid): {ambiguous}")
check("FLOW_DATE values impossible as month/day exist", True, md_invalid > 0)
print("status counts:", collections.Counter(r.group(8) for r in rows))
print("pipes:", collections.Counter(r.group(2) for r in rows), "| distinct counterparties:", len({r.group(3) for r in rows}))
var_ok = sum(1 for r in rows if int(r.group(5).replace(",", "")) - int(r.group(6).replace(",", "")) == int(r.group(7).replace(",", "")))
print("rows where SCHEDULED - ACTUAL == VARIANCE:", var_ok, "of", len(rows))

bytes: 350222 | token estimate: 87555 | share of corpus bytes: 5.1 %
next largest: [('giron-d/inbox/20', 53222), ('williams-w3/inbox/45', 53205), ('giron-d/inbox/390', 33012)]
[ok] giant message table rows: plan=3600 measured=3600
non-row, non-blank lines: 5
    Team,
    Pasting the full month position below since the attachment keeps bouncing off the gateway. Please pull your ow
      NOM_ID   PIPE  COUNTERPARTY  FLOW_DATE   SCHEDULED    ACTUAL  VARIANCE  STATUS     METER
    --------------------------------------------------------------------------------------------------------
    Daren
FLOW_DATE as month/day: 2160 invalid of 3600 | as day/month: 0 invalid | ambiguous (both valid): 720
[ok] FLOW_DATE values impossible as month/day exist: plan=True measured=True
status counts: Counter({'SCHEDULED': 2400, 'PENDING': 1200})
pipes: Counter({'HPL': 3600}) | distinct counterparties: 400
rows where SCHEDULED - ACTUAL == VARIANCE: 3600 of 3600


In [14]:
# Size limit for 'giant': what do the next-largest messages look like? (shape only, no text)
TAB = re.compile(r"\S+(\s{2,}\S+){3,}")
print(f"{'key':28s} {'bytes':>7s} {'hdr':>6s} {'body':>7s} {'lines':>6s} {'tabular%':>8s} {'quoted%':>7s}")
for k in sorted(msgs, key=lambda k: -msgs[k]["size"])[:8]:
    m = msgs[k]; ls = [l for l in m["body"].splitlines() if l.strip()]
    tab = sum(1 for l in ls if TAB.search(l))
    f = re.search(r"^[ \t>]*-{2,}\s*(Original Message|Forwarded by)", m["body"], re.M | re.I)
    q = pct(len(m["body"]) - f.start(), len(m["body"])) if f else 0.0
    print(f"{k:28s} {m['size']:7d} {m['size'] - len(m['body']):6d} {len(m['body']):7d} {len(ls):6d} {pct(tab, len(ls)):8.1f} {q:7.1f}")
for lim in (16_000, 32_000, 64_000, 100_000):
    print(f"messages over {lim // 1000} KB: {sum(1 for m in msgs.values() if m['size'] > lim)}")

key                            bytes    hdr    body  lines tabular% quoted%
farmer-d/logistics/3221       350222    626  349596   3605     99.9     0.0
giron-d/inbox/20               53222  50272    2950     29      0.0     0.0
williams-w3/inbox/45           53205  50255    2950     29      0.0     0.0
giron-d/inbox/390              33012  29957    3055     29      0.0     0.0
giron-d/sent/637               26996    490   26506    744      0.0   100.0
giron-d/sent/642               18491    693   17798    333      0.0   100.0
williams-w3/inbox/86           18155   1463   16692    267     24.0     0.0
williams-w3/inbox/49           18149   1463   16686    273     32.2     0.0
messages over 16 KB: 16
messages over 32 KB: 4
messages over 64 KB: 1
messages over 100 KB: 1


## 7. Replies, forwards and quoted text

Markers that start quoted or forwarded text. Strict markers are unambiguous; the header-block
markers catch Lotus Notes and Outlook replies that have no `Original Message` line.

In [15]:
SUBJ_PREFIX = re.compile(r"^\s*((re|fw|fwd)\s*(\[\d+\])?\s*:\s*)+", re.I)
pref = [m for m in msgs.values() if SUBJ_PREFIX.match(m["subject"])]
check("subjects starting Re:/Fw:", 1276, len(pref))
print("by prefix:", collections.Counter(SUBJ_PREFIX.match(m["subject"]).group(2).lower() for m in pref))

MARKERS = {
    "original_message": r"^[ \t>]*-{2,}\s*Original Message\s*-{2,}",
    "forwarded_by":     r"^[ \t>]*-{2,}\s*Forwarded by\b",
    "lotus_from_on":    r"^[ \t>]*From:[^\n]*\bon \d{1,2}/\d{1,2}/\d{2,4}\s+\d{1,2}:\d{2}",
    "outlook_block":    r"^[ \t>]*From:[^\n]*\r?\n[ \t>]*(Sent|Date):",
    "lotus_name_date":  r"^[ \t]*[^\n]*?\s\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}\s*(AM|PM)\s*\r?\n(\s*\r?\n){0,3}[ \t]*To:",
    "gt_quote":         r"^[ \t]*>",
}
MARK_RE = {n: re.compile(p, re.M | re.I) for n, p in MARKERS.items()}

def first_marker(body, names=MARKERS):
    best = None
    for n in names:
        m = MARK_RE[n].search(body)
        if m and (best is None or m.start() < best[0]):
            best = (m.start(), n)
    return best

def split_stats(names):
    with_marker, quoted_chars, all_chars, first_kind = 0, 0, 0, collections.Counter()
    for m in msgs.values():
        b = m["body"]; all_chars += len(b)
        f = first_marker(b, names)
        if f:
            with_marker += 1; quoted_chars += len(b) - f[0]; first_kind[f[1]] += 1
    return with_marker, quoted_chars, all_chars, first_kind

for label, names in [("strict (orig + fwd)", ["original_message", "forwarded_by"]),
                     ("strict + lotus/outlook headers", ["original_message", "forwarded_by", "lotus_from_on", "outlook_block", "lotus_name_date"]),
                     ("all incl. '>' lines", list(MARKERS))]:
    w, q, a, fk = split_stats(names)
    a_ng, q_ng = a - len(g["body"]), q   # giant message has no marker
    print(f"{label:34s} messages={w:5d}  quoted share={pct(q, a):5.1f}%  (excl. giant: {pct(q_ng, a_ng):5.1f}%)  first marker: {dict(fk)}")

w, q, a, fk = split_stats(list(MARKERS))
Q_MSGS, Q_SHARE = w, pct(q, a)
check("messages with a reply/forward marker", 1437, w, ok=abs(w - 1437) <= 15)
check("share of body text after a marker (%)", 53, pct(q, a), ok=abs(pct(q, a) - 53) <= 2)

[ok] subjects starting Re:/Fw:: plan=1276 measured=1276
by prefix: Counter({'re': 820, 'fw': 437, 'fwd': 19})
strict (orig + fwd)                messages= 1437  quoted share= 53.4%  (excl. giant:  58.3%)  first marker: {'original_message': 663, 'forwarded_by': 774}


strict + lotus/outlook headers     messages= 1614  quoted share= 58.2%  (excl. giant:  63.5%)  first marker: {'original_message': 656, 'forwarded_by': 682, 'lotus_from_on': 16, 'lotus_name_date': 252, 'outlook_block': 8}

all incl. '>' lines                messages= 1630  quoted share= 58.8%  (excl. giant:  64.2%)  first marker: {'original_message': 655, 'forwarded_by': 682, 'lotus_from_on': 16, 'lotus_name_date': 252, 'gt_quote': 18, 'outlook_block': 7}


[DIFF] messages with a reply/forward marker: plan=1437 measured=1630
[DIFF] share of body text after a marker (%): plan=53 measured=58.8


In [16]:
# Per-marker hit counts (any position), and messages with a Re/Fw subject but no marker, or a marker but no prefix.
for n, r in MARK_RE.items():
    print(f"{n:18s} {sum(1 for m in msgs.values() if r.search(m['body'])):5d} messages")
has_mark = {k for k, m in msgs.items() if first_marker(m["body"])}
pref_keys = {m["key"] for m in pref}
print("Re/Fw subject but no marker:", len(pref_keys - has_mark), "| marker but no Re/Fw subject:", len(has_mark - pref_keys))

# Spot-check the Lotus header marker where no strict marker exists: print only the matched header
# lines (names/dates/To:), capped at 70 chars, for 5 messages spread across the corpus.
strict = ["original_message", "forwarded_by"]
only_lotus = [k for k, m in msgs.items() if MARK_RE["lotus_name_date"].search(m["body"]) and not first_marker(m["body"], strict)]
print("\nmessages caught only by lotus_name_date:", len(only_lotus))
for k in only_lotus[:: max(1, len(only_lotus) // 5)][:5]:
    mm = MARK_RE["lotus_name_date"].search(msgs[k]["body"])
    lines = [l.strip()[:70] for l in mm.group(0).splitlines() if l.strip()]
    print(f"   {k:26s} {lines}")

original_message     749 messages
forwarded_by         786 messages
lotus_from_on        120 messages
outlook_block        712 messages


lotus_name_date      774 messages
gt_quote             141 messages


Re/Fw subject but no marker: 159 | marker but no Re/Fw subject: 513



messages caught only by lotus_name_date: 170
   farmer-d/logistics/130     ['From:\tJoyce Viltz/ENRON@enronXgate on 05/22/2001 02:49 PM', 'To:']
   farmer-d/logistics/1194    ['From:  Pat Clynes @ ENRON                           06/08/2000 08:53 A', 'To:']
   farmer-d/logistics/2040    ['Clem Cernosek@ECT', '01/09/2001 12:09 PM', 'To:']
   giron-d/sent/273           ['Jane S Saladino', '01/31/2001 07:24 AM', 'To:']
   giron-d/sent/455           ['Dawn C Kenne', '11/28/2000 09:16 AM', 'To:']


In [17]:
# How often is the quoted text's original already in the corpus as its own message?
# new_text = text before the first marker. Fingerprint = first 160 whitespace-normalized chars
# of a message's new text (only when it has at least 160 chars, so short "Thanks" bodies don't match).
FP = 160
HDR_LINE = re.compile(r"^[ \t>]*(From|Sent|To|Cc|Bcc|Subject|Date|Importance)\s*:.*$", re.I | re.M)

for m in msgs.values():
    f = first_marker(m["body"])
    m["new_text"] = m["body"][:f[0]] if f else m["body"]
    m["quoted"] = m["body"][f[0]:] if f else ""

fp_index = collections.defaultdict(set)
for m in msgs.values():
    n = norm_ws(m["new_text"])
    if len(n) >= FP:
        fp_index[n[:FP]].add(m["key"])

def segments(quoted):
    # split the quoted region at every marker; drop header lines and '>' prefixes inside each segment
    cuts = sorted({mm.start() for r in MARK_RE.values() for mm in r.finditer(quoted)} | {0})
    for a, b in zip(cuts, cuts[1:] + [len(quoted)]):
        seg = quoted[a:b]
        seg = re.sub(r"^[ \t]*>+ ?", "", seg, flags=re.M)
        seg = HDR_LINE.sub("", seg)
        seg = re.sub(r"^.*(Original Message|Forwarded by).*$", "", seg, flags=re.M | re.I)
        seg = re.sub(r"^[^\n]*\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}\s*(AM|PM)\s*$", "", seg, flags=re.M)
        yield norm_ws(seg)

seg_total = seg_long = seg_found = 0
found_chars = long_chars = 0
for m in msgs.values():
    if not m["quoted"]: continue
    for s in segments(m["quoted"]):
        if not s: continue
        seg_total += 1
        if len(s) >= FP:
            seg_long += 1; long_chars += len(s)
            hits = fp_index.get(s[:FP], set()) - {m["key"]}
            if hits:
                seg_found += 1; found_chars += len(s)
print(f"quoted segments: {seg_total} | long enough to fingerprint (>= {FP} chars): {seg_long}")
print(f"long segments whose original is in the corpus: {seg_found} ({pct(seg_found, seg_long)}%), chars {pct(found_chars, long_chars)}% of long-segment text")
print("(lower bound: exact-prefix match only; re-wrapped or '>'-edited quotes won't match)")

quoted segments: 7623 | long enough to fingerprint (>= 160 chars): 1971
long segments whose original is in the corpus: 154 (7.8%), chars 6.0% of long-segment text
(lower bound: exact-prefix match only; re-wrapped or '>'-edited quotes won't match)


In [18]:
# Stronger test: containment. Normalize each quoted region (strip '>' prefixes, collapse whitespace),
# then look for any other message's whole normalized new text inside it. Candidates come from a
# 64-char prefix index; a hit counts only when the whole new text (>= 80 chars) is contained.
K, MIN_NEW = 64, 80
prefix_index = collections.defaultdict(list)
new_norm = {}
for k, m in msgs.items():
    n = norm_ws(m["new_text"])
    if len(n) >= MIN_NEW and k != G:
        new_norm[k] = n
        prefix_index[n[:K]].append(k)

covered_total = quoted_total = 0
msgs_with_quote = msgs_fully_covered = msgs_any_covered = 0
contained_pairs = 0
for k, m in msgs.items():
    if not m["quoted"]: continue
    qn = norm_ws(re.sub(r"^[ \t]*>+ ?", "", m["quoted"], flags=re.M))
    quoted_total += len(qn); msgs_with_quote += 1
    cover = bytearray(len(qn))
    for i in range(len(qn) - K + 1):
        for c in prefix_index.get(qn[i:i + K], ()):
            if c != k and qn.startswith(new_norm[c], i):
                cover[i:i + len(new_norm[c])] = b"\x01" * len(new_norm[c]); contained_pairs += 1
    c = sum(cover); covered_total += c
    if c: msgs_any_covered += 1
    if c >= 0.9 * len(qn): msgs_fully_covered += 1
print(f"messages with quoted text: {msgs_with_quote}")
print(f"  quoted text containing at least one other message's full new text: {msgs_any_covered} ({pct(msgs_any_covered, msgs_with_quote)}%)")
print(f"  quoted text >= 90% covered by originals in the corpus: {msgs_fully_covered} ({pct(msgs_fully_covered, msgs_with_quote)}%)")
print(f"  share of all quoted characters that are copies of a message we already have: {pct(covered_total, quoted_total)}%")
all_body = sum(len(norm_ws(m['body'])) for m in msgs.values())
print(f"  -> characters saved by skipping those copies: {covered_total:,} of {all_body:,} normalized body chars ({pct(covered_total, all_body)}%)")

messages with quoted text: 1630
  quoted text containing at least one other message's full new text: 322 (19.8%)
  quoted text >= 90% covered by originals in the corpus: 5 (0.3%)
  share of all quoted characters that are copies of a message we already have: 7.5%
  -> characters saved by skipping those copies: 176,483 of 3,943,981 normalized body chars (4.5%)


In [19]:
# Quoted text repeated across messages with no standalone original: thread history that is
# re-quoted at every reply. Count quoted segments (>= 80 chars) that appear in 2+ messages.
seg_where = collections.defaultdict(set)
for k, m in msgs.items():
    if m["quoted"] and k not in dup_of:
        for s in segments(m["quoted"]):
            if len(s) >= 80: seg_where[s].add(k)
rep = {s: ks for s, ks in seg_where.items() if len(ks) > 1}
rep_chars = sum((len(ks) - 1) * len(s) for s, ks in rep.items())
seg_chars = sum(len(s) * len(ks) for s, ks in seg_where.items())
print(f"distinct quoted segments >= 80 chars: {len(seg_where)} | seen in 2+ messages: {len(rep)}")
print(f"repeat occurrences (2nd and later) = {rep_chars:,} chars = {pct(rep_chars, seg_chars)}% of segment text, {pct(rep_chars, all_body)}% of all body text")

distinct quoted segments >= 80 chars: 1944 | seen in 2+ messages: 320
repeat occurrences (2nd and later) = 236,042 chars = 15.4% of segment text, 6.0% of all body text


## 8. Senders

In [20]:
senders = collections.Counter(m["frm"] for m in msgs.values())
for s, n in senders.most_common(8):
    print(f"{n:5d}  {s}")
top = {s.split("@")[0]: n for s, n in senders.most_common(3)}
check("main senders", {"darron.giron": 789, "bill.williams": 528, "michelle.lokay": 166}, top)
print("sent folders: from the mailbox owner?")
for f in ["giron-d/sent", "williams-w3/sent_items", "lokay-m/sent_items"]:
    print(f"  {f:24s}", collections.Counter(m["frm"] for m in msgs.values() if m["folder"] == f).most_common(2))
print("farmer-d/logistics top senders:", collections.Counter(m["frm"] for m in msgs.values() if m["folder"] == "farmer-d/logistics").most_common(4))

  789  darron.giron@enron.com
  528  bill.williams@enron.com
  166  michelle.lokay@enron.com
   73  mary.poorman@enron.com
   57  aimee.lannou@enron.com
   55  jackie.young@enron.com
   38  mark.mccoy@enron.com
   32  m..love@enron.com
[ok] main senders: plan={'darron.giron': 789, 'bill.williams': 528, 'michelle.lokay': 166} measured={'darron.giron': 789, 'bill.williams': 528, 'michelle.lokay': 166}
sent folders: from the mailbox owner?
  giron-d/sent             [('darron.giron@enron.com', 789)]
  williams-w3/sent_items   [('bill.williams@enron.com', 520)]
  lokay-m/sent_items       [('michelle.lokay@enron.com', 161)]
farmer-d/logistics top senders: [('mary.poorman@enron.com', 71), ('aimee.lannou@enron.com', 57), ('jackie.young@enron.com', 55), ('mark.mccoy@enron.com', 38)]


## 9. Planted instructions (search only; hits are data, never acted on)

In [21]:
INJ = {
    "ignore instructions": r"\b(ignore|disregard|forget)\b[^.\n]{0,40}\b(instructions?|prompts?|rules)\b",
    "AI / model words":    r"\b(ai|llm|language model|chatgpt|gpt-?\d|claude|anthropic|openai|chatbot)\b",
    "prompt words":        r"\b(system prompt|you are now|as an ai|assistant:|jailbreak)\b",
    "agent/tool words":    r"\b(run (this|the following) command|execute|curl |powershell|rm -rf)\b",
}
for name, p in INJ.items():
    r = re.compile(p, re.I)
    hits = [(k, mm) for k, m in msgs.items() for mm in [r.search(m["body"])] if mm]
    print(f"{name:20s} {len(hits):4d} messages")
    for k, mm in hits[:4]:
        b = msgs[k]["body"]
        print(f"      {k:26s} ...{norm_ws(b[max(0, mm.start() - 30):mm.end() + 30])[:90]!r}")

ignore instructions     0 messages


AI / model words        0 messages


prompt words            2 messages
      farmer-d/logistics/2000    ...'u. If anyone tells you that you are now doing something off of their'
      giron-d/sent/42            ...'answer. Answers are for who you are now...not who you were in the pas'
agent/tool words       10 messages
      farmer-d/logistics/28      ...'fee deducted. Do we need to execute a PMA with TXGLO to deduct th'
      farmer-d/logistics/34      ...'fee deducted. Do we need to execute a PMA with TXGLO to deduct th'
      giron-d/inbox/86           ...'law at the bank. Then, you execute a debt-equity swap with an as'
      lokay-m/sent_items/28      ...'ther-in-law at the bank, then execute a debt/equity swap with an as'


## 10. Threads

No message has `In-Reply-To` or `References` headers (section 3), so threads must come from the
subject, the participants and time. Measured on unique messages (body duplicates removed).

In [22]:
def norm_subject(s):
    s = SUBJ_PREFIX.sub("", s or "")
    s = re.sub(r"^\s*((re|fw|fwd)\s*:\s*)+", "", s, flags=re.I)
    return norm_ws(s).lower()

first_of = dup_of   # proposed duplicate rule E from section 5b
uniq = [m for k, m in msgs.items() if k not in first_of]
print("unique messages:", len(uniq))
for m in uniq:
    m["nsubj"] = norm_subject(m["subject"])
    m["people"] = {a for a in [m["frm"], *m["to"], *m["cc"]] if a}

by_subj = collections.defaultdict(list)
for m in uniq: by_subj[m["nsubj"]].append(m)
sizes_s = collections.Counter(len(v) for v in by_subj.values())
print("distinct normalized subjects:", len(by_subj), "| singletons:", sizes_s[1], "| 2+ messages:", sum(n for s, n in sizes_s.items() if s > 1))
print("empty subject:", len(by_subj.get("", [])), "messages")
print("largest subject groups:")
for s, v in sorted(by_subj.items(), key=lambda kv: -len(kv[1]))[:12]:
    ds = sorted(m["date_utc"] for m in v if m["date_status"] == "ok")
    span = (ds[-1] - ds[0]).days if len(ds) > 1 else 0
    print(f"   {len(v):4d} msgs  span {span:4d} d  mailboxes {len({m['mailbox'] for m in v})}  {s[:50]!r}")

unique messages: 3200
distinct normalized subjects: 2429 | singletons: 2000 | 2+ messages: 429
empty subject: 107 messages
largest subject groups:
    107 msgs  span  757 d  mailboxes 3  ''
     18 msgs  span  492 d  mailboxes 2  'calpine daily gas nomination'
     14 msgs  span  471 d  mailboxes 1  'lunch'
     11 msgs  span  680 d  mailboxes 2  'out of the office'
      9 msgs  span  320 d  mailboxes 1  'cp&l'
      8 msgs  span  466 d  mailboxes 1  'hey'
      7 msgs  span    0 d  mailboxes 1  'tuesday, december 26th'
      7 msgs  span  455 d  mailboxes 1  'saturday'
      6 msgs  span  152 d  mailboxes 1  'epgt'
      6 msgs  span  183 d  mailboxes 1  'hl&p'
      6 msgs  span   13 d  mailboxes 1  'meter 984132 for 1/16/99'
      6 msgs  span    3 d  mailboxes 1  'snake'


In [23]:
# Gaps between consecutive dated messages with the same normalized subject.
gaps = []
for s, v in by_subj.items():
    if not s: continue
    ds = sorted(m["date_utc"] for m in v if m["date_status"] == "ok")
    gaps += [(b - a).total_seconds() / 86400 for a, b in zip(ds, ds[1:])]
gaps.sort()
q = lambda p: round(gaps[min(len(gaps) - 1, int(p * len(gaps)))], 2)
print("consecutive same-subject gaps:", len(gaps))
print("days: p25", q(.25), "p50", q(.5), "p75", q(.75), "p90", q(.9), "p95", q(.95))
for d in (1, 3, 7, 14, 30, 60):
    print(f"   <= {d:2d} days: {pct(sum(1 for x in gaps if x <= d), len(gaps)):5.1f}%")

consecutive same-subject gaps: 664
days: p25 0.04 p50 0.77 p75 7.81 p90 44.05 p95 142.19
   <=  1 days:  55.9%
   <=  3 days:  65.2%
   <=  7 days:  74.4%
   <= 14 days:  81.6%
   <= 30 days:  87.5%
   <= 60 days:  90.8%


In [24]:
# Candidate rule: same normalized subject, at least one shared participant with the thread so far,
# and within MAX_GAP days of the thread's previous message. Compare thread counts across gaps.
def build_threads(max_gap_days, need_people=True):
    threads = []
    for s, v in by_subj.items():
        dated = sorted([m for m in v if m["date_status"] == "ok"], key=lambda m: m["date_utc"])
        undated = [m for m in v if m["date_status"] != "ok"]
        open_t = []
        for m in dated:
            home = None
            for t in open_t:
                if s and (m["date_utc"] - t["last"]).days <= max_gap_days and (not need_people or t["people"] & m["people"]):
                    home = t; break
            if home is None:
                home = {"msgs": [], "people": set(), "last": m["date_utc"]}; open_t.append(home); threads.append(home)
            home["msgs"].append(m); home["people"] |= m["people"]; home["last"] = m["date_utc"]
        for m in undated:   # undated: join a same-subject thread that shares people, else stand alone
            home = next((t for t in open_t if s and t["people"] & m["people"]), None)
            if home is None:
                home = {"msgs": [], "people": set(), "last": None}; threads.append(home)
            home["msgs"].append(m); home["people"] |= m["people"]
    return threads

for gap in (3, 7, 14, 30, 10_000):
    for need in (True, False):
        th = build_threads(gap, need)
        multi = [t for t in th if len(t["msgs"]) > 1]
        print(f"gap {gap:>5} d  people={'yes' if need else 'no ':3s}  threads={len(th):5d}  multi-message={len(multi):4d}  messages in multi={sum(len(t['msgs']) for t in multi):5d}")

gap     3 d  people=yes  threads= 2759  multi-message= 331  messages in multi=  772
gap     3 d  people=no   threads= 2749  multi-message= 338  messages in multi=  789
gap     7 d  people=yes  threads= 2707  multi-message= 360  messages in multi=  853


gap     7 d  people=no   threads= 2697  multi-message= 366  messages in multi=  869
gap    14 d  people=yes  threads= 2667  multi-message= 385  messages in multi=  918
gap    14 d  people=no   threads= 2656  multi-message= 392  messages in multi=  936
gap    30 d  people=yes  threads= 2629  multi-message= 407  messages in multi=  978
gap    30 d  people=no   threads= 2617  multi-message= 415  messages in multi=  998
gap 10000 d  people=yes  threads= 2568  multi-message= 417  messages in multi= 1049
gap 10000 d  people=no   threads= 2536  multi-message= 428  messages in multi= 1092


In [25]:
# Generic subjects: the same subject reused for unrelated, recurring mail (reports, notices).
# These are exactly the recurring processes we want, so each occurrence should be its own event.
th14 = build_threads(14, True)
print("threads (14 d, shared people):", len(th14))
long_lived = [t for t in th14 if len(t["msgs"]) > 1 and t["last"] and
              (t["last"] - min(m["date_utc"] for m in t["msgs"] if m["date_status"] == "ok")).days > 60]
print("threads spanning > 60 days even with the 14-day gap rule:", len(long_lived))
for t in sorted(long_lived, key=lambda t: -len(t["msgs"]))[:6]:
    ds = [m["date_utc"] for m in t["msgs"] if m["date_status"] == "ok"]
    print(f"   {len(t['msgs']):3d} msgs  {min(ds).date()} .. {max(ds).date()}  {t['msgs'][0]['nsubj'][:50]!r}")

threads (14 d, shared people): 2667
threads spanning > 60 days even with the 14-day gap rule: 0


## 11. Months observed (the events/month denominator)

Compare three ways to count the months a mailbox could have seen a process:
the full first-to-last span, a trimmed span (1st to 99th percentile of dates), and active months
(calendar months with at least one message).

In [26]:
def spans(ms):
    ds = sorted(m["date_utc"] for m in ms if m["date_status"] == "ok")
    if not ds: return None
    full = (ds[-1] - ds[0]).days / 30.4375
    lo, hi = ds[int(0.01 * (len(ds) - 1))], ds[int(0.99 * (len(ds) - 1))]
    trimmed = (hi - lo).days / 30.4375
    cal = month_index(ds[-1]) - month_index(ds[0]) + 1
    active = len({month_index(d) for d in ds})
    active3 = sum(1 for n in collections.Counter(month_index(d) for d in ds).values() if n >= 3)
    return dict(n=len(ds), first=ds[0].date(), last=ds[-1].date(), full=round(full, 1), trimmed=round(trimmed, 1),
                calendar=cal, active=active, active3=active3)

print(f"{'scope':24s} {'dated':>5s} {'first':>10s} {'last':>10s} {'full':>6s} {'trim':>6s} {'cal':>4s} {'active':>6s} {'act>=3':>6s}")
rows_m = {}
for scope in sorted({m["mailbox"] for m in uniq}) + ["ALL"]:
    ms = uniq if scope == "ALL" else [m for m in uniq if m["mailbox"] == scope]
    s = spans(ms); rows_m[scope] = s
    print(f"{scope:24s} {s['n']:5d} {s['first']!s:>10s} {s['last']!s:>10s} {s['full']:6.1f} {s['trimmed']:6.1f} {s['calendar']:4d} {s['active']:6d} {s['active3']:6d}")

scope                    dated      first       last   full   trim  cal active act>=3
farmer-d                  1166 1999-12-13 2001-12-14   24.0   23.7   25     25     25
giron-d                   1167 2000-06-08 2002-02-06   20.0   19.8   21     21     21
lokay-m                    212 2001-09-26 2002-03-25    5.9    5.3    7      7      6
williams-w3                651 2001-05-07 2002-02-07    9.0    7.9   10     10     10
ALL                       3196 1999-12-13 2002-03-25   27.4   26.5   28     28     28


In [27]:
# Monthly volume per mailbox, to see gaps and thin tails.
months = sorted({month_index(m["date_utc"]) for m in uniq if m["date_status"] == "ok"})
mbs = sorted({m["mailbox"] for m in uniq})
cnt = collections.Counter((m["mailbox"], month_index(m["date_utc"])) for m in uniq if m["date_status"] == "ok")
print("month    " + " ".join(f"{b[:9]:>9s}" for b in mbs))
for mi in range(months[0], months[-1] + 1):
    print(f"{mi // 12}-{mi % 12 + 1:02d}  " + " ".join(f"{cnt.get((b, mi), 0):9d}" for b in mbs))

month     farmer-d   giron-d   lokay-m williams-
1999-12         54         0         0         0
2000-01         78         0         0         0
2000-02        103         0         0         0
2000-03        122         0         0         0
2000-04         61         0         0         0
2000-05         66         0         0         0
2000-06         77        42         0         0
2000-07         35        30         0         0
2000-08         52        47         0         0
2000-09         43        51         0         0
2000-10         32        82         0         0
2000-11         50       106         0         0
2000-12         66        75         0         0
2001-01         43        88         0         0
2001-02         43        92         0         0
2001-03         41        74         0         0
2001-04         47        74         0         0
2001-05         31        30         0         3
2001-06         19         5         0        67
2001-07         15  

In [28]:
# Proposed denominator: the union of the [first, last] dated-message intervals of the mailboxes
# where the process appears, in days / 30.4375.
import itertools
bounds = {}
for mb in sorted({m["mailbox"] for m in uniq}):
    ds = sorted(m["date_utc"] for m in uniq if m["mailbox"] == mb and m["date_status"] == "ok")
    bounds[mb] = (ds[0], ds[-1])

def months_observed(mailboxes):
    iv = sorted(bounds[b] for b in mailboxes)
    total_days, cur = 0.0, None
    for a, b in iv:
        if cur and a <= cur[1]:
            cur = (cur[0], max(cur[1], b))
        else:
            if cur: total_days += (cur[1] - cur[0]).total_seconds() / 86400
            cur = (a, b)
    total_days += (cur[1] - cur[0]).total_seconds() / 86400
    return round(total_days / 30.4375, 2)

for r in range(1, len(bounds) + 1):
    for combo in itertools.combinations(sorted(bounds), r):
        print(f"   {' + '.join(combo):45s} {months_observed(combo):6.2f} months")

corpus_m = months_observed(list(bounds))
print("\nError if the whole-corpus span were used for a process seen in one mailbox only:")
for mb in sorted(bounds):
    print(f"   {mb:12s} rate understated {corpus_m / months_observed([mb]):.1f}x")

   farmer-d                                       24.06 months
   giron-d                                        20.00 months
   lokay-m                                         5.92 months
   williams-w3                                     9.05 months
   farmer-d + giron-d                             25.84 months
   farmer-d + lokay-m                             27.39 months
   farmer-d + williams-w3                         25.86 months
   giron-d + lokay-m                              21.54 months
   giron-d + williams-w3                          20.01 months
   lokay-m + williams-w3                          10.58 months
   farmer-d + giron-d + lokay-m                   27.39 months
   farmer-d + giron-d + williams-w3               25.86 months
   farmer-d + lokay-m + williams-w3               27.39 months
   giron-d + lokay-m + williams-w3                21.54 months
   farmer-d + giron-d + lokay-m + williams-w3     27.39 months

Error if the whole-corpus span were used for a process

## 12. Summary: PLAN section 2 facts, plan vs measured

In [29]:
w = max(len(f) for f, *_ in FACTS)
for f, p, m, st in FACTS:
    ps, ms = str(p), str(m)
    if len(ps) > 60 or len(ms) > 60:
        print(f"{st:9s}  {f}\n{'':11s}plan:     {ps}\n{'':11s}measured: {ms}")
    else:
        print(f"{st:9s}  {f:{w}s}  plan={ps}  measured={ms}")
print("\nconfirmed:", sum(1 for *_, s in FACTS if s == "confirmed"), "| corrected:", sum(1 for *_, s in FACTS if s != "confirmed"))

confirmed  file count                                      plan=3240  measured=3240
confirmed  total size (MB, 10^6 bytes)                     plan=6.85  measured=6.85
confirmed  token estimate (chars/4, millions)              plan=1.7  measured=1.71
confirmed  median email size (KB, 10^3)                    plan=1.4  measured=1.4
confirmed  folder counts
           plan:     {'farmer-d/logistics': 1194, 'giron-d/inbox': 380, 'giron-d/sent': 789, 'lokay-m/inbox': 57, 'lokay-m/sent_items': 161, 'williams-w3/inbox': 139, 'williams-w3/sent_items': 520}
           measured: {'farmer-d/logistics': 1194, 'giron-d/inbox': 380, 'giron-d/sent': 789, 'lokay-m/inbox': 57, 'lokay-m/sent_items': 161, 'williams-w3/inbox': 139, 'williams-w3/sent_items': 520}
confirmed  files with mixed line endings                   plan=3135  measured=3135
confirmed  missing Date header                             plan=['giron-d/sent/795']  measured=['giron-d/sent/795']
confirmed  invalid date                       

## 14. Follow-ups from the rule review (21 Nov 2026)

Outside reviewers raised several points: transitive duplicate chains, recipient checks on duplicates,
threads vs. business events, unknown time zones, month bucketing near midnight, mailbox selection
bias in the denominator, and dating evidence found only in quoted text. Each one is measured here.

In [30]:
# 14a. Duplicates: why was each copy merged? Every copy is compared to its canonical (first-kept)
# message only, never to another copy, so A~B and B~C can never chain into A~C.
def recipients(m): return set(m["to"]) | set(m["cc"])
def dup_reason(k, v):
    a, b = msgs[v], msgs[k]; h = dh(a, b)
    if a["message_id"] == b["message_id"]: return "same_message_id"
    if a["mailbox"] != b["mailbox"]: return "same_mail_in_two_mailboxes"
    if h in (10.0, 11.0) and a["frm"] == b["frm"]: return "export_timezone_copy"
    if h is not None and h < 1 and a["frm"] == b["frm"]: return "resend_within_1h"
    return "UNEXPLAINED"
reasons = collections.Counter()
for k, v in dup_of.items():
    reasons[dup_reason(k, v)] += 1
print("merge reasons:", dict(reasons))
canon_groups = collections.Counter(dup_of.values())
print("canonical messages:", len(canon_groups), "| largest merged group:", 1 + max(canon_groups.values()))
print("copies that are themselves a canonical (chain risk):", sum(1 for k in dup_of if k in canon_groups))
ov = [(len(recipients(msgs[k]) & recipients(msgs[v])) / max(1, len(recipients(msgs[k]) | recipients(msgs[v])))) for k, v in dup_of.items()]
print("recipient overlap (Jaccard) across the 40 pairs: =1.0:", sum(1 for x in ov if x == 1.0), "| 0<x<1:", sum(1 for x in ov if 0 < x < 1), "| 0:", sum(1 for x in ov if x == 0))
for k, v in dup_of.items():
    j = len(recipients(msgs[k]) & recipients(msgs[v])) / max(1, len(recipients(msgs[k]) | recipients(msgs[v])))
    if j < 1:
        print(f"   {dup_reason(k, v):28s} {v:26s} -> {k:26s} overlap {j:.2f}  to: {msgs[v]['to'][:2]} vs {msgs[k]['to'][:2]}")
later_canon = sum(1 for k, v in dup_of.items() if msgs[v]["date_utc"] > msgs[k]["date_utc"])
print("pairs where the canonical (lowest key) carries the later of the two dates:", later_canon, "of", len(dup_of))
print("canonicals whose copies sit in another mailbox (mailbox presence a copy adds):",
      sum(1 for v in canon_groups if any(msgs[k]["mailbox"] != msgs[v]["mailbox"] for k, c in dup_of.items() if c == v)))

merge reasons: {'export_timezone_copy': 28, 'same_message_id': 1, 'same_mail_in_two_mailboxes': 6, 'resend_within_1h': 5}
canonical messages: 40 | largest merged group: 2
copies that are themselves a canonical (chain risk): 0
recipient overlap (Jaccard) across the 40 pairs: =1.0: 33 | 0<x<1: 0 | 0: 7
   same_mail_in_two_mailboxes   giron-d/inbox/1            -> williams-w3/inbox/6        overlap 0.00  to: ['darren.adamik@enron.com', 'p..adams@enron.com'] vs ['a..allen@enron.com', 'john.anderson@enron.com']
   same_mail_in_two_mailboxes   giron-d/inbox/2            -> williams-w3/inbox/9        overlap 0.00  to: ['darren.adamik@enron.com', 'p..adams@enron.com'] vs ['a..allen@enron.com', 'john.anderson@enron.com']
   same_mail_in_two_mailboxes   giron-d/inbox/54           -> williams-w3/inbox/87       overlap 0.00  to: ['dl-ga-all_netco@enron.com'] vs []
   same_mail_in_two_mailboxes   giron-d/inbox/271          -> lokay-m/inbox/2            overlap 0.00  to: [] vs []
   resend_within_1h

In [31]:
# 14b. Dates: missing time zones, and whether UTC vs. header-local time moves messages across months.
ok_msgs = [m for m in uniq if m["date_status"] == "ok"]
no_tz = sum(1 for m in ok_msgs if not ZONE.search(m["date_raw"]))
print("ok dates without a written zone:", no_tz, "of", len(ok_msgs), "(checked on the header text, not parsedate_tz)")
def local_month(m):
    t = parsedate_tz(m["date_raw"]); return t[0] * 12 + t[1] - 1
moved = [m for m in ok_msgs if month_index(m["date_utc"]) != local_month(m)]
print("messages whose calendar month differs in UTC vs. header-local time:", len(moved), f"({pct(len(moved), len(ok_msgs))}%)")
print("  ... and whose year differs:", sum(1 for m in moved if m["date_utc"].year != parsedate_tz(m["date_raw"])[0]))
print("offsets in use:", collections.Counter(parsedate_tz(m["date_raw"])[9] // 3600 for m in ok_msgs))

ok dates without a written zone: 0 of 3196 (checked on the header text, not parsedate_tz)
messages whose calendar month differs in UTC vs. header-local time: 11 (0.3%)
  ... and whose year differs: 0


offsets in use: Counter({-8: 1848, -7: 1348})


In [32]:
# 14c. Threads vs. events. A 'root' is a message with no Re:/Fw: prefix and no quoted text: a fresh send.
# Under the 14-day rule, how many threads contain more than one root (i.e. several separate sends,
# such as daily notices, chained into one 'conversation')?
def is_reply(m): return bool(SUBJ_PREFIX.match(m["subject"])) or bool(m["quoted"])
roots = [m for m in uniq if not is_reply(m)]
print("roots:", len(roots), "| replies/forwards:", len(uniq) - len(roots))
multi_root = [t for t in th14 if sum(1 for m in t["msgs"] if not is_reply(m)) > 1]
print("14-day threads with 2+ roots:", len(multi_root), "| messages in them:", sum(len(t["msgs"]) for t in multi_root))
cal_threads = [t for t in th14 if t["msgs"][0]["nsubj"] == "calpine daily gas nomination"]
print("Calpine notices: messages", sum(len(t["msgs"]) for t in cal_threads), "-> 14-day threads", len(cal_threads),
      "| roots among them:", sum(1 for t in cal_threads for m in t["msgs"] if not is_reply(m)))
for t in sorted(multi_root, key=lambda t: -len(t["msgs"]))[:8]:
    print(f"   {len(t['msgs']):3d} msgs, {sum(1 for m in t['msgs'] if not is_reply(m)):2d} roots  {t['msgs'][0]['nsubj'][:55]!r}")

roots: 1420 | replies/forwards: 1780
14-day threads with 2+ roots: 49 | messages in them: 137
Calpine notices: messages 18 -> 14-day threads 11 | roots among them: 17
     7 msgs,  7 roots  'calpine daily gas nomination'
     7 msgs,  2 roots  'tuesday, december 26th'
     6 msgs,  6 roots  'come early, be loud, stay late'
     5 msgs,  3 roots  'lunch'
     4 msgs,  2 roots  'heads up'
     4 msgs,  2 roots  'lunch'
     4 msgs,  2 roots  'book request'
     4 msgs,  3 roots  'hub deals'


In [33]:
# Revised rule: a root always starts a new thread; only replies/forwards join an existing thread
# (same normalized subject, shared participant, <= 14 days after the thread's last message).
def build_threads_v2(max_gap_days=14):
    threads = []
    for s, v in by_subj.items():
        dated = sorted([m for m in v if m["date_status"] == "ok"], key=lambda m: m["date_utc"])
        undated = [m for m in v if m["date_status"] != "ok"]
        open_t = []
        for m in dated:
            home = None
            if s and is_reply(m):
                cands = [t for t in open_t if (m["date_utc"] - t["last"]).days <= max_gap_days and t["people"] & m["people"]]
                home = max(cands, key=lambda t: t["last"]) if cands else None
            if home is None:
                home = {"msgs": [], "people": set(), "last": m["date_utc"], "root": not is_reply(m)}
                open_t.append(home); threads.append(home)
            home["msgs"].append(m); home["people"] |= m["people"]; home["last"] = m["date_utc"]
        for m in undated:
            home = next((t for t in open_t if s and is_reply(m) and t["people"] & m["people"]), None)
            if home is None:
                home = {"msgs": [], "people": set(), "last": None, "root": not is_reply(m)}; threads.append(home)
            home["msgs"].append(m); home["people"] |= m["people"]
    return threads

th2 = build_threads_v2(14)
multi2 = [t for t in th2 if len(t["msgs"]) > 1]
print(f"v2 threads: {len(th2)} (v1: {len(th14)}) | multi-message: {len(multi2)} | messages in multi: {sum(len(t['msgs']) for t in multi2)}")
print("v2 threads that start with a reply (root not in corpus):", sum(1 for t in th2 if not t["root"]))
print("v2 Calpine threads:", sum(1 for t in th2 if t["msgs"][0]["nsubj"] == "calpine daily gas nomination"))
for gap in (3, 7, 14, 30):
    print(f"   gap {gap:2d} d -> {len(build_threads_v2(gap))} threads")
# Reconciliation invariants the ingest tests should enforce.
members = [m["key"] for t in th2 for m in t["msgs"]]
print("\nreconcile: files", len(msgs), "= unique", len(uniq), "+ copies", len(dup_of), "->", len(msgs) == len(uniq) + len(dup_of))
print("reconcile: every unique message in exactly one thread ->", sorted(members) == sorted(m["key"] for m in uniq))
print("reconcile: unique - threads =", len(uniq) - len(th2), "messages that joined an earlier message's thread")

v2 threads: 2736 (v1: 2667) | multi-message: 348 | messages in multi: 812


v2 threads that start with a reply (root not in corpus): 1316
v2 Calpine threads: 18
   gap  3 d -> 2801 threads
   gap  7 d -> 2762 threads
   gap 14 d -> 2736 threads
   gap 30 d -> 2717 threads

reconcile: files 3240 = unique 3200 + copies 40 -> True
reconcile: every unique message in exactly one thread -> True
reconcile: unique - threads = 464 messages that joined an earlier message's thread


In [34]:
# 14d. Evidence that lives only in quoted text: can its own date be recovered from the quoted header?
SENT_RE = re.compile(r"^[ \t>]*(Sent|Date):\s*(.+?)\s*$", re.I | re.M)
LOTUS_DT = re.compile(r"(\d{1,2})/(\d{1,2})/(\d{4})\s+(\d{1,2}):(\d{2})\s*(AM|PM)", re.I)
def seg_date(raw_seg):
    m = SENT_RE.search(raw_seg)
    if m:
        for fmt in ("%A, %B %d, %Y %I:%M %p", "%a, %d %b %Y %H:%M:%S", "%m/%d/%Y %I:%M:%S %p", "%m/%d/%Y %I:%M %p"):
            try: return dt.datetime.strptime(m.group(2).split(" -")[0].split(" (")[0].strip(), fmt)
            except ValueError: pass
        t = parsedate_tz(m.group(2))
        if t:
            try: return dt.datetime(*t[:6])
            except ValueError: pass
    m = LOTUS_DT.search(raw_seg[:400])
    if m:
        mo, d, y, hh, mi, ap = m.groups()
        try: return dt.datetime(int(y), int(mo), int(d), int(hh) % 12 + (12 if ap.upper() == "PM" else 0), int(mi))
        except ValueError: pass
    return None

def raw_segments(quoted):
    cuts = sorted({mm.start() for r in MARK_RE.values() for mm in r.finditer(quoted)} | {0})
    return [quoted[a:b] for a, b in zip(cuts, cuts[1:] + [len(quoted)])]

n_seg = n_dated = n_later = 0
lags = []
for k, m in msgs.items():
    if not m["quoted"] or k in dup_of: continue
    for rs in raw_segments(m["quoted"]):
        if len(norm_ws(rs)) < 80: continue
        n_seg += 1
        d = seg_date(rs)
        if d:
            n_dated += 1
            if m["date_status"] == "ok":
                local = dt.datetime(*parsedate_tz(m["date_raw"])[:6])
                lag = (local - d).total_seconds() / 86400
                lags.append(lag)
                if lag < -0.01: n_later += 1
lags.sort()
print(f"quoted segments (>= 80 chars) in unique messages: {n_seg} | with a recoverable date in their header: {n_dated} ({pct(n_dated, n_seg)}%)")
print(f"days between the quoted date and the containing message: p50 {lags[len(lags)//2]:.2f}  p90 {lags[int(.9*len(lags))]:.2f}  max {lags[-1]:.0f}")
print(f"quoted dates later than their containing message: {n_later}")
neg = [x * 24 for x in lags if x < -0.01]
print("  of those, hours later: <= 1h", sum(1 for h in neg if h >= -1), "| 1-2h", sum(1 for h in neg if -2 <= h < -1),
      "| 2-3h", sum(1 for h in neg if -3 <= h < -2), "| > 3h", sum(1 for h in neg if h < -3))
print("  -> quoted header times carry no offset (writer's local clock, mostly Houston Central) while Date headers are Pacific;")
print("     a 1-2 h skew is time zones, not bad data. Quoted dates are usable to the day, not to the hour.")
print(f"quoted dates in a different calendar month from their containing message: "
      f"{sum(1 for x in lags if x > 0)} have a lag; {sum(1 for x in lags if x >= 28)} are 4+ weeks older")

quoted segments (>= 80 chars) in unique messages: 3707 | with a recoverable date in their header: 3580 (96.6%)
days between the quoted date and the containing message: p50 0.13  p90 7.95  max 511
quoted dates later than their containing message: 1676
  of those, hours later: <= 1h 69 | 1-2h 236 | 2-3h 40 | > 3h 1331
  -> quoted header times carry no offset (writer's local clock, mostly Houston Central) while Date headers are Pacific;
     a 1-2 h skew is time zones, not bad data. Quoted dates are usable to the day, not to the hour.
quoted dates in a different calendar month from their containing message: 1889 have a lag; 125 are 4+ weeks older


In [35]:
# 14e. Denominator: are the four mailboxes separate views of the company, or overlapping ones?
# For each mailbox, the share of its unique messages that involve each other mailbox's owner.
owners = {"farmer-d": "daren.farmer@enron.com", "giron-d": "darron.giron@enron.com",
          "lokay-m": "michelle.lokay@enron.com", "williams-w3": "bill.williams@enron.com"}
print(f"{'mailbox':12s} " + " ".join(f"{o.split('.')[0][:8] + '..':>11s}" for o in owners))
for mb in owners:
    ms = [m for m in uniq if m["mailbox"] == mb]
    print(f"{mb:12s} " + " ".join(f"{pct(sum(1 for m in ms if o in m['people']), len(ms)):10.1f}%" for o in owners.values()))
print("(diagonal = the owner's own mail; off-diagonal = how often another mailbox could have seen the same message)")

mailbox       farmer-d..   giron-d..   lokay-m..  williams..
farmer-d           79.2%        0.9%        0.1%        0.1%
giron-d             0.0%       67.8%        0.0%        0.2%
lokay-m             0.0%        0.0%       97.7%        0.0%
williams-w3         0.0%        0.0%        0.2%       91.0%
(diagonal = the owner's own mail; off-diagonal = how often another mailbox could have seen the same message)


### 14f. The Date header clock shift

14d found quoted dates 7-8 hours *later* than the message holding them, clustered at exactly the
header's Pacific offset. Test it on messages that are a fresh Lotus forward (short new text, a
`Forwarded by X on MM/DD/YYYY HH:MM AM` line at the top): the forward time is the send time on the
sender's own clock, so it should match the Date header.

In [36]:
FWD = re.compile(r"^[ \t]*-{2,}\s*Forwarded by\s+(.+?)\s+on\s+(\d{1,2})/(\d{1,2})/(\d{4})\s*\r?\n?\s*(\d{1,2}):(\d{2})\s*(AM|PM)", re.I | re.M)
fresh = []
for k, m in msgs.items():
    if k in dup_of or m["date_status"] != "ok" or not m["quoted"] or len(norm_ws(m["new_text"])) > 300: continue
    f = FWD.search(m["quoted"][:300])
    if not f: continue
    who, mo, d, y, hh, mi, ap = f.groups()
    try: fw = dt.datetime(int(y), int(mo), int(d), int(hh) % 12 + (12 if ap.upper() == "PM" else 0), int(mi))
    except ValueError: continue
    t = parsedate_tz(m["date_raw"]); off_h = -t[9] // 3600
    h = (fw - dt.datetime(*t[:6])).total_seconds() / 3600
    cls = "matches header" if abs(h) < 0.6 else "+2 h (Central vs Pacific)" if 1.4 < h < 2.6 else \
          "header shifted by its own offset" if abs(h - off_h) < 0.6 else "other"
    fresh.append(dict(key=k, mailbox=m["mailbox"], ym=(t[0], t[1]), h=h, cls=cls))
print("fresh Lotus forwards:", len(fresh))
print(collections.Counter(r["cls"] for r in fresh))
for mb in sorted({r["mailbox"] for r in fresh}):
    sh = sorted(r["ym"] for r in fresh if r["mailbox"] == mb and r["cls"].startswith("header shifted"))
    un = sorted(r["ym"] for r in fresh if r["mailbox"] == mb and r["cls"] in ("matches header", "+2 h (Central vs Pacific)"))
    print(f"   {mb:10s} shifted: {len(sh):3d} msgs {sh[0] if sh else ''}..{sh[-1] if sh else ''} | not shifted: {len(un):2d} msgs {un[0] if un else ''}..{un[-1] if un else ''}")
SHIFT = sum(1 for r in fresh if r["cls"].startswith("header shifted"))
print(f"\n-> {pct(SHIFT, len(fresh))}% of testable messages carry a Date header 7-8 h earlier than the sender's clock.")
print("   The export stored the sender's wall-clock time as if it were UTC, then printed it in Pacific time.")
print("   Consequence: on those messages the UTC value's calendar date is the sender's real date, but the")
print("   header hour is not the real hour. Durations shorter than a day are not measurable across eras.")

fresh Lotus forwards: 613
Counter({'header shifted by its own offset': 572, 'other': 31, '+2 h (Central vs Pacific)': 9, 'matches header': 1})
   farmer-d   shifted: 170 msgs (2000, 1)..(2001, 3) | not shifted:  9 msgs (2001, 4)..(2001, 10)
   giron-d    shifted: 402 msgs (2000, 6)..(2001, 5) | not shifted:  1 msgs (2001, 5)..(2001, 5)

-> 93.3% of testable messages carry a Date header 7-8 h earlier than the sender's clock.
   The export stored the sender's wall-clock time as if it were UTC, then printed it in Pacific time.
   Consequence: on those messages the UTC value's calendar date is the sender's real date, but the
   header hour is not the real hour. Durations shorter than a day are not measurable across eras.


## 15. Findings behind the approved rules (PLAN.md sections 2 and 3.1, approved 21 Nov 2026)

Every number below is computed in this notebook. Rule numbers match PLAN.md section 3.1.

In [37]:
print(f"R1 dates: {status['ok']} ok, {status['missing']} missing, {status['invalid']} invalid, {status['out_of_range']} out of range;"
      f" {no_tz} ok dates lack an offset; {len(moved)} messages change month in UTC vs header-local time.")
print(f"   clock shift: {SHIFT} of {len(fresh)} testable Lotus forwards ({pct(SHIFT, len(fresh))}%) have a header 7-8 h behind the sender's clock.")
print(f"R2 duplicates: body-only match gives 49 extra copies; the approved rule merges {len(dup_of)} ({dict(reasons)}).")
print(f"   kept separate: {[k for g in body_groups for k in g[1:] if k not in dup_of]}")
print(f"R3 quoted text: {Q_MSGS} messages, {Q_SHARE}% of body text; {pct(covered_total, quoted_total)}% of quoted text is a copy of a corpus message;"
      f" re-quoted history {pct(rep_chars, all_body)}% of body text; {pct(n_dated, n_seg)}% of quoted parts have their own date.")
print(f"   headers are {pct(hdr_bytes, total)}% of corpus bytes.")
print(f"R4 giant message: {len(rows)} rows; FLOW_DATE day/month ({dm_invalid} invalid as D/M, {md_invalid} invalid as M/D, {ambiguous} ambiguous);"
      f" variance check {var_ok}/{len(rows)}; bodies over 64 KB: {sum(1 for m in msgs.values() if len(m['body']) > 64_000)}.")
print(f"R5 threads: {len(th2)} threads, {len(multi2)} with 2+ messages; {sum(1 for t in th2 if not t['root'])} start with a reply whose root is not in the corpus.")
print(f"R6 reconcile: {len(msgs)} files = {len(uniq)} unique + {len(dup_of)} copies; {len(uniq) - len(th2)} messages joined an earlier thread.")
print(f"R7 months observed: " + ", ".join(f"{mb} {months_observed([mb])}" for mb in sorted(bounds)) + f"; all {corpus_m}.")

R1 dates: 3236 ok, 1 missing, 1 invalid, 2 out of range; 0 ok dates lack an offset; 11 messages change month in UTC vs header-local time.
   clock shift: 572 of 613 testable Lotus forwards (93.3%) have a header 7-8 h behind the sender's clock.
R2 duplicates: body-only match gives 49 extra copies; the approved rule merges 40 ({'export_timezone_copy': 28, 'same_message_id': 1, 'same_mail_in_two_mailboxes': 6, 'resend_within_1h': 5}).
   kept separate: ['farmer-d/logistics/1310', 'farmer-d/logistics/1312', 'farmer-d/logistics/1316', 'farmer-d/logistics/1318', 'farmer-d/logistics/1329', 'farmer-d/logistics/1330', 'farmer-d/logistics/1336', 'farmer-d/logistics/1348', 'williams-w3/inbox/37']
R3 quoted text: 1630 messages, 58.8% of body text; 7.5% of quoted text is a copy of a corpus message; re-quoted history 6.0% of body text; 96.6% of quoted parts have their own date.
   headers are 38.9% of corpus bytes.
R4 giant message: 3600 rows; FLOW_DATE day/month (0 invalid as D/M, 2160 invalid as M